# 04 Data cleaning

Validate and clean all five released TRACE-X tables. The [schema summary](../reports/schema_summary.md) supplies the released columns and keys. The [datasheet](../Phase1_Dataset/DATASHEET.md) and [configuration](../Phase1_Dataset/code/config.yaml) define categories and domains. The [training EDA](../reports/eda_findings.md) found no missing values in the seven approved inputs and flagged some otherwise valid weights. Those observations support preserving the readings and leaving imputation for a later phase.

## Cleaning decisions and reasons

| Issue | Rule and reason |
| --- | --- |
| Storage types | Use the released numeric, Boolean and identifier types. Accept safe numeric casts and unambiguous Boolean encodings. Retain identifiers as text so their spelling and leading zeros stay intact. Store dates as native dates and the event timestamp as a timezone-naive timestamp at its original second precision. |
| Missing values | Retain GPS dropout, missing fraud type on genuine rows, and entity status or certificate fields that the release leaves blank. These gaps carry no verified replacement value. Normalize optional blank text and floating-point NaNs to null. Stop on a missing required value, including a label or key. |
| Duplicate records | Compare every repeated key against the corresponding full raw rows. Keep the first only when the records are identical. Stop on conflicting records with the same key because the evidence cannot tell us which is correct. |
| Invalid timestamps | Validate exact ISO spelling and calendar dates. Reject malformed, shifted, or offset-bearing input. The release has no timezone to infer or missing event time to fill. |
| Invalid values | Check fixed constraints such as coordinate limits, audit scores from 0 to 100, valid sequence indices, ordered storage limits and expiry on or after production. Reject known violations with the affected table and column. Keep statistical outliers, temperature excursions, failed checks, broken seals, and suspicious or masked cases. |
| Missing foreign keys | Require batch references to resolve to real medicine and manufacturer records, and require fraud and transaction references to resolve to real batches. Stop on unresolved links so a dependent row does not silently disappear or acquire a different parent. |
| Inconsistent categories | Map casing and surrounding whitespace to documented spellings for known categories. Report unknown categories as errors. The release offers no evidence for mapping an unknown category to an existing one. |

These rules use fixed schemas and documented domains. No imputation value, clipping threshold, category vocabulary, or other data-dependent parameter is learned. Every split uses the same rules. The cleaned Parquet tables are an archival copy. Later feature assembly must still follow the [feature audit](../reports/feature_audit.csv) and use the binary target separately.

In [ ]:
from hashlib import file_digest
from pathlib import Path
import json
import sys
from tempfile import TemporaryDirectory

import pandas as pd
import pyarrow as pa
import pyarrow.compute as pc
import pyarrow.parquet as pq
import yaml
from IPython.display import display

current = Path.cwd()
root = next(path for path in (current, *current.parents) if (path / "AGENTS.md").is_file())
sys.path.insert(0, str(root))
sys.dont_write_bytecode = True
from src.cleaning import CleaningError, KEYS, category_domains, clean_block, clean_parquet, validate_schema

dataset = root / "Phase1_Dataset" / "dataset"
config = yaml.safe_load((root / "Phase1_Dataset" / "code" / "config.yaml").read_text(encoding="utf-8"))
domains = category_domains(config)
processed = root / "processed"
cleaned_dir = processed / "cleaned"
boundary = json.loads((processed / "split_boundaries.json").read_text(encoding="utf-8"))
manifest_path = processed / boundary["assignment_file"]
with manifest_path.open("rb") as source:
    assert file_digest(source, "sha256").hexdigest() == boundary["assignment_sha256"]
key = boundary["transaction_key"]
clock = boundary["prediction_timestamp"]
target = boundary["target_column"]
assert key == KEYS["transactions"] and clock == "timestamp"
print("Loaded the fixed cleaning domains and verified the saved split manifest.")

## Check how the rules respond to bad data

Exercise the reusable functions on small test inputs. These checks cover safe conversion, category spelling, optional missing data, malformed calendar dates and Boolean values, absent columns, unsupported categories, orphaned keys, and exact versus conflicting duplicates. These examples run in a temporary directory.

In [ ]:
def expect_failure(action, expected_text):
    try:
        action()
    except CleaningError as error:
        assert expected_text in str(error), str(error)
    else:
        raise AssertionError(f"Expected a cleaning error containing {expected_text!r}")


def block(data):
    return pa.record_batch(data)


category = clean_block("medicines", block({"drug_type": ["  VaCcInE  "], "storage_min": ["2"]}), domains)
assert category["drug_type"].to_pylist() == ["vaccine"] and category["storage_min"].to_pylist() == [2]
assert clean_block("transactions", block({"qr_verified": [" FALSE ", "1"]}), domains)["qr_verified"].to_pylist() == [False, True]
assert clean_block("transactions", block({"gps_lat": [float("nan")]}), domains)["gps_lat"].null_count == 1
expect_failure(lambda: clean_block("transactions", block({"timestamp": ["2025-02-30T12:00:00"]}), domains), "invalid ISO")
expect_failure(lambda: clean_block("transactions", block({"timestamp": ["2025-01-01T12:00:00Z"]}), domains), "invalid ISO")
expect_failure(lambda: clean_block("transactions", block({"batch_id": [None]}), domains), "missing required")
expect_failure(lambda: clean_block("transactions", block({"door_open": ["maybe"]}), domains), "Boolean")
expect_failure(lambda: clean_block("medicines", block({"storage_min": [2.5]}), domains), "lossy")
expect_failure(lambda: clean_block("medicines", block({"drug_type": ["ointment"]}), domains), "unknown categories")
expect_failure(lambda: validate_schema("medicines", pa.schema([("medicine_id", pa.string())])), "missing columns")

with TemporaryDirectory(dir=processed) as folder:
    example_dir = Path(folder)
    source = example_dir / "example.parquet"
    output = example_dir / "cleaned.parquet"
    medicine = pq.read_table(dataset / "medicines.parquet").slice(0, 1)
    pq.write_table(pa.concat_tables([medicine, medicine]), source)
    check = clean_parquet("medicines", source, output, domains, batch_size=1)
    assert check["rows_before"] == 2 and check["rows_after"] == check["verified_duplicates_removed"] == 1
    price = medicine.schema.get_field_index("price")
    conflict = medicine.set_column(price, "price", pa.array([medicine["price"][0].as_py() + 1]))
    pq.write_table(pa.concat_tables([medicine, conflict]), source)
    expect_failure(lambda: clean_parquet("medicines", source, output, domains, batch_size=1), "conflicting records")

    batch = pq.read_table(dataset / "batches.parquet").slice(0, 1)
    pq.write_table(batch, source)
    expect_failure(
        lambda: clean_parquet("batches", source, output, domains,
                              references={"medicine_id": pa.array([], type=pa.large_string())}),
        "missing parent IDs",
    )
print("All example validation assertions passed.")

## Clean the five tables and verify the saved split

Clean parents first so the next table can check its references against canonical IDs. Transactions are read in batches. An initial key pass checks for repeated records across batches. Stage the cleaned files and publish them after verifying the row counts, the binary target, and the fixed transaction keys and timestamps.

In [ ]:
def clean_release(staging):
    results = {}

    def clean(name, references=None):
        path = staging / f"{name}.parquet"
        results[name] = clean_parquet(name, dataset / path.name, path, domains, references)
        assert pq.read_metadata(path).num_rows == results[name]["rows_after"]
        return path

    entities_path = clean("entities")
    medicines_path = clean("medicines")
    entities = pq.read_table(entities_path, columns=["entity_id", "entity_type"])
    manufacturers = entities.filter(pc.equal(entities["entity_type"], "manufacturer"))["entity_id"]
    medicine_ids = pq.read_table(medicines_path, columns=["medicine_id"])["medicine_id"]
    batches_path = clean("batches", {"manufacturer_id": manufacturers, "medicine_id": medicine_ids})
    batch_ids = pq.read_table(batches_path, columns=["batch_id"])["batch_id"]
    fraud_path = clean("fraud_events", {"batch_id": batch_ids})
    assert pq.read_metadata(fraud_path).num_rows == len(batch_ids), "Fraud records must cover every batch"
    transactions_path = clean("transactions", {"batch_id": batch_ids})
    assert results["transactions"]["rows_after"] == boundary["source_rows"]

    # Validate the label artifact row by row without inspecting feature-target relationships.
    target_path = processed / "transaction_target.parquet"
    with pq.ParquetFile(transactions_path) as transactions, pq.ParquetFile(target_path) as labels:
        for actual, frozen in zip(
            transactions.iter_batches(columns=[*key, target], batch_size=50_000),
            labels.iter_batches(columns=[*key, target], batch_size=50_000),
            strict=True,
        ):
            assert all(actual[column].equals(frozen[column]) for column in key)
            assert pc.cast(actual[target], pa.uint8()).equals(frozen[target])

    # Check the manifest against the exact transaction identity and timestamp, including excluded rows.
    actual = pq.read_table(transactions_path, columns=[*key, clock, "batch_tx_index"]).to_pandas(types_mapper=pd.ArrowDtype)
    assert not actual.duplicated(["batch_id", "batch_tx_index"]).any()
    fixed = pq.read_table(manifest_path, columns=[*key, clock, "split"]).to_pandas(types_mapper=pd.ArrowDtype)
    membership = fixed.merge(actual, on=[*key, clock], how="inner", validate="one_to_one")
    assert len(membership) == len(fixed) == len(actual)
    split_counts = membership["split"].value_counts().to_dict()
    assert split_counts == {name: data["rows"] for name, data in boundary["split_summary"].items()}
    return results, split_counts


cleaned_dir.mkdir(exist_ok=True)
with TemporaryDirectory(dir=cleaned_dir) as folder:
    staging = Path(folder)
    audit, split_counts = clean_release(staging)
    for name in audit:
        path = staging / f"{name}.parquet"
        path.replace(cleaned_dir / path.name)

print("Five cleaned Parquet files saved. All frozen split rows and binary labels verified.")

## Report the cleaning results

The first table records row counts and changes. Type conversions count non-null values whose stored type changed. The second table identifies affected columns and records preserved nulls. These are structural counts. Split membership and the separate target remain available from the existing artifacts.

In [ ]:
row_report = pd.DataFrame([
    {
        "table": name,
        "rows_before": details["rows_before"],
        "rows_after": details["rows_after"],
        "duplicates_removed": details["verified_duplicates_removed"],
        "typed_values": sum(details["type_converted_rows"].values()),
        "category_repairs": sum(details["normalized_category_values"].values()),
        "missing_markers_normalized": sum(details["missing_markers_to_null"].values()),
        "nulls_preserved": sum(details["nulls_preserved"].values()),
    }
    for name, details in audit.items()
]).set_index("table")
display(row_report)
assert row_report["rows_before"].sub(row_report["rows_after"]).eq(row_report["duplicates_removed"]).all()

audit_rows = []
for name, details in audit.items():
    for action in ("type_converted_rows", "normalized_category_values", "missing_markers_to_null", "nulls_preserved"):
        for column, count in details[action].items():
            if count:
                audit_rows.append((name, column, action, count))
display(pd.DataFrame(audit_rows, columns=["table", "column", "action", "values"]))
display(pd.Series(split_counts, name="verified transaction rows").rename_axis("split").to_frame())
print("All cleaning and validation assertions passed.")